In [ ]:
import pandas as pd 
import numpy as np
from IPython.display import display
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu
import missingno as msno
%matplotlib inline
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
#from functions.transformers import *
from functions.eda_functions import test_mcar, list_missing_values

In [ ]:
#Cargar datos
rawTrain = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
#1. se conserva la feature target para el analisis de missing values
useTrain = rawTrain.copy()
print(f'dimensiones de nuestro df: {useTrain.shape}')

In [ ]:
#Analisis de features con missing values
list_missing_values(useTrain)

In [ ]:
#PoolQC: Calidad de la piscina.
print(f"-- PoolQC ---")
#Visualización de los datos faltantes, cantidad y porcentaje
print(useTrain['PoolQC'].unique())
missing_poolqc = pd.DataFrame(useTrain['PoolQC'].value_counts(dropna=False))
missing_poolqc['percentage(%)'] = np.round(missing_poolqc['count'] / useTrain.shape[0] * 100, 2)
display(missing_poolqc)


PoolQC_missingValuesNumber = useTrain[useTrain['PoolArea'] == 0]['PoolQC'].isna().sum()
print(f'El numero de missing values corresponde a {PoolQC_missingValuesNumber} que es igual al numero de PoolArea = 0')

#Imputación de los datos faltantes
useTrain.loc[:, 'PoolQC'] = useTrain['PoolQC'].fillna('NA')
#Visualización de la relación entre PoolQC y PoolArea graficamente
plt.figure(figsize=(8,4))
sns.violinplot(x='PoolQC', y='PoolArea', data=useTrain)
plt.title('Pool Quality vs Pool Area')
plt.show()
#PoolQC: Calidad de la piscina. depende de PoolArea. Si PoolArea es 0, PoolQC es NA.
#MNAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#MiscFeature: Característica miscelánea no cubierta en otras categorías.
print(f"-- MiscFeature ---")
#MiscVal: Valor monetario de la característica miscelánea.
#Visualización de los datos faltantes, cantidad y porcentaje
missing_miscFeature = pd.DataFrame(useTrain['MiscFeature'].value_counts(dropna=False))
missing_miscFeature['percentage(%)'] = np.round(missing_miscFeature['count'] / useTrain.shape[0] * 100, 2)
display(missing_miscFeature)

#visualizar las filas con datos faltantes en MiscFeature y su relación con MiscVal
MiscFeature_missingValuesNumber = useTrain[useTrain['MiscVal'] == 0]['MiscFeature'].isna().sum()
print(f'El numero de missing values corresponde a {MiscFeature_missingValuesNumber} que es igual al numero de MiscVal = 0')
#Imputación de los datos faltantes
useTrain.loc[:, 'MiscFeature'] = useTrain['MiscFeature'].fillna('NA')
#Visualizacion de la relación entre MiscFeature y MiscVal graficamente
plt.figure(figsize=(8,4))
sns.violinplot(x='MiscFeature', y='MiscVal', data=useTrain)
plt.title('Misc Feature vs Misc Val')
plt.show()
#MiscFeature: Característica miscelánea no cubierta en otras categorías. Depende de MiscVal. Si MiscVal es 0, MiscFeature es NA.
#MNAR Las features estan relacionadas. Se puede imputar. No son aleatorias.

In [ ]:
#Alley: Tipo de acceso por callejón a la propiedad.
print(f"-- Alley ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_alley = pd.DataFrame(useTrain['Alley'].value_counts(dropna=False))
missing_alley['percentage(%)'] = np.round(missing_alley['count'] / useTrain.shape[0] * 100, 2)
#display(missing_alley)

#visualizar las filas con datos faltantes en Alley
test_mcar(useTrain, 'Alley')
#Se diferencia los precios de las casas con y sin callejón NA. tienen un mas elevado. se imputan con NA
useTrain.loc[:, 'Alley'] = useTrain['Alley'].fillna('NA')

In [ ]:
#Fence: Calidad de la cerca alrededor de la propiedad.
print(f"-- Fence ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_fence = pd.DataFrame(useTrain['Fence'].value_counts(dropna=False))
missing_fence['percentage(%)'] = np.round(missing_fence['count'] / useTrain.shape[0] * 100, 2)
display(missing_fence)
test_mcar(useTrain, 'Fence', 'LotArea')
#Se imputan con NA. Se diferencia los precios de las casas con y sin cerca NA. pero ademas un area de lote mas grande.
useTrain.loc[:, 'Fence'] = useTrain['Fence'].fillna('NA')

In [ ]:
#MasVnrArea: Área de revestimiento de mampostería.
missing_masvnrarea = pd.DataFrame(useTrain['MasVnrArea'].isnull().value_counts(dropna=False))
missing_masvnrarea['percentage(%)'] = np.round(missing_masvnrarea['count'] / useTrain.shape[0] * 100, 2)
display(missing_masvnrarea)

useTrain.loc[:, 'MasVnrArea'] = useTrain['MasVnrArea'].fillna(0)
#Se imputan los missing values con 0.
#El tipo de revestimiento de mampostería (MasVnrType) es missing value, por lo que el área de revestimiento de mampostería (MasVnrArea) es 0. Se puede imputar con 0. No son aleatorias.

In [ ]:
#MasVnrType: Tipo de revestimiento de mampostería.
print(f"-- MasVnrType ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_masvnrtype = pd.DataFrame(useTrain['MasVnrType'].value_counts(dropna=False))
missing_masvnrtype['percentage(%)'] = np.round(missing_masvnrtype['count'] / useTrain.shape[0] * 100, 2)
display(missing_masvnrtype)

test_mcar(useTrain, 'MasVnrType', 'MasVnrArea')
#La densidad de area construida 0, es donde se concentran la mayoría de Area Nula, con excepción de algunos casos.
#Dato que todos los MasVnrArea = 0, MasVnrType = None. Se puede imputar con None. No son aleatorias.
useTrain.loc[:, 'MasVnrType'] = useTrain['MasVnrType'].fillna('None')
display(useTrain[(useTrain['MasVnrType'] == 'None') & (useTrain['MasVnrArea'] != 0)].groupby(['MasVnrType', 'Exterior1st', 'Exterior2nd'])['MasVnrArea'].count().sort_values(ascending=False))

#Viendo la comparativa de revestimientos con el agrupamiendo, se decide dejar estos en None ya que los tipos no se encuentran en la lista de tipos de revestimiento. Se puede imputar con None. No son aleatorias.

In [ ]:
#FireplaceQu: Calidad de la chimenea.
print(f"-- FireplaceQu ---")
#Visualización de los datos faltantes, cantidad y porcentaje
missing_fireplacequ = pd.DataFrame(useTrain['FireplaceQu'].value_counts(dropna=False))
missing_fireplacequ['percentage(%)'] = np.round(missing_fireplacequ['count'] / useTrain.shape[0] * 100, 2)
display(missing_fireplacequ)
#Se da por hecho que si Fireplaces es 0, entonces FireplaceQu es NA
condition_fireplaceQU = (useTrain['Fireplaces'] == 0)
useTrain.loc[condition_fireplaceQU, 'FireplaceQu'] = 'NA';
#Se imputa bajo la condicion que si Fireplaces es 0, entonces FireplaceQu es NA. No son aleatorias.

In [ ]:
list_missing_values(useTrain)

In [ ]:
#GarageArea: Tamaño del garaje en pies cuadrados.
#GarageFinish: Acabado interior del garaje.
#GarageQual: Calidad del garaje.
#GarageType: Ubicación del garaje.
#GarageCond: Condición del garaje.
#GarageYrBlt: Año de construcción del garaje.
condition_garageArea = (useTrain['GarageArea'] == 0)
#Siempre que GarageArea es 0, entonces GarageFinish, GarageQual, GarageType, GarageCond y GarageYrBlt son NA. Se puede imputar con NA. No son aleatorias.
useTrain.loc[condition_garageArea, 'GarageFinish'] = 'NA'
useTrain.loc[condition_garageArea, 'GarageQual'] = 'NA'
useTrain.loc[condition_garageArea, 'GarageType'] = 'NA'
useTrain.loc[condition_garageArea, 'GarageCond'] = 'NA'
useTrain.loc[condition_garageArea, 'GarageYrBlt'] = 0
useTrain[(useTrain['GarageArea'] == 0)][['GarageArea', 'GarageFinish', 'GarageQual', 'GarageType', 'GarageCond', 'GarageYrBlt']]

In [ ]:
list_missing_values(useTrain)

In [ ]:
#BsmtFinType2: Calificación del área terminada del sótano (si hay varios tipos).
#BsmtExposure: Se refiere a las paredes del sótano a nivel de jardín o con salida.
#BsmtCond: Evalúa la condición general del sótano.
#BsmtQual: Evalúa la altura del sótano.
#BsmtFinType1: Calificación del área terminada del sótano.
condition_bsmt = (useTrain['TotalBsmtSF'] == 0)
useTrain.loc[condition_bsmt, 'BsmtFinType1'] = 'NA'
useTrain.loc[condition_bsmt, 'BsmtFinType2'] = 'NA'
useTrain.loc[condition_bsmt, 'BsmtExposure'] = 'NA'
useTrain.loc[condition_bsmt, 'BsmtCond'] = 'NA'
useTrain.loc[condition_bsmt, 'BsmtQual'] = 'NA'
useTrain[(useTrain['TotalBsmtSF'] == 0)][['TotalBsmtSF', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure', 'BsmtCond', 'BsmtQual']]

In [ ]:
list_missing_values(useTrain)